In [48]:
import sys
import os
import urllib3

# Add your local ThreatConnect SDK to path
sys.path.append("Z:/HTOC/Data_Analytics/threatconnect")
from ThreatConnect import ThreatConnect
from RequestObject import RequestObject

# Add your project repo to path
project_root = r"C:\Users\jaskew\Documents\project_repository\scripts\Data Movement\ThrearConnect-api-pull"
if project_root not in sys.path:
    sys.path.append(project_root)

from utils.config_loader import load_config

# Load API config
config_path = os.path.join(project_root, "utils", "config.json")
try:
    api_secret_key, api_access_id, api_base_url, api_default_org = load_config(config_path)
    print(f"Loaded config from: {config_path}")
    print(f"Base URL: {api_base_url}")
    print(f"Access ID: {api_access_id}")
    print(f"Default Org: {api_default_org}")
except Exception as e:
    print(f"[ERROR] Failed to load configuration: {e}")
    sys.exit(1)

# Disable SSL verification warnings (use cautiously)
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# Initialize ThreatConnect session
try:
    tc = ThreatConnect(api_access_id, api_secret_key, api_default_org, api_base_url)
    print("ThreatConnect initialized.")
except Exception as e:
    print(f"[ERROR] Failed to initialize ThreatConnect: {e}")
    sys.exit(1)

# Define the owner (organization scope)
owner = 'HTOC Org'

# Create a request object to fetch indicators (or other data)
try:
    ro = RequestObject()
    ro.set_http_method('GET')
    ro.set_owner(owner)
    ro.set_owner_allowed(True)
    # ro.set_resource_pagination(True)  # Uncomment if needed
    print("RequestObject successfully created.")
except Exception as e:
    print(f"[ERROR] Failed to initialize RequestObject: {e}")
    sys.exit(1)

Loaded config from: C:\Users\jaskew\Documents\project_repository\scripts\Data Movement\ThrearConnect-api-pull\utils\config.json
Base URL: https://hvs.threatconnect.com/api
Access ID: 09783848890162390382
Default Org: HTOC Org
ThreatConnect initialized.
RequestObject successfully created.


In [49]:
import pandas as pd
from datetime import datetime, timedelta
import pytz

# Define time period
# Calculate the start date (2 days ago) at 00:00:00 UTC
start_date = (datetime.now(pytz.UTC) - timedelta(days=2)).date()

# Format as 'YYYY-MM-DDT00:00:00Z'
start = f"{start_date}T00:00:00Z"

# List of owners to pull from
import urllib.parse

list_of_owners = ['HTOC Org']
final_results = []
type_names = [
    "Address", "EmailAddress", "File", "Host", "URL", "ASN", "CIDR", "Email Subject", "Hashtag", "Mutex", "Registry Key", "Stripped URL", "User Agent"]
type_name_condition = ", ".join([f'"{t}"' for t in type_names])

for owner in list_of_owners:
    print(f"\nQuerying owner: {owner}")
    try:
        tql_raw = (
            f'ownerName EQ "{owner}" AND '
            f'typeName IN ({type_name_condition})'
            f'lastObserved >= "{start}"'
        )
        tql_encoded = urllib.parse.quote(tql_raw)

        ro.set_http_method('GET')
        ro.set_request_uri(
            f'/v3/indicators?tql={tql_encoded}&fields=tags,observations,associatedGroups&resultStart=0&resultLimit=10000'
        )

         # Send the request
        response = tc.api_request(ro)

        # Parse response
        if response.headers.get('content-type') == 'application/json':
            results = response.json()
            final_results.append(results)
        else:
            print(f"Unexpected response format: {response.headers.get('content-type')}")

    except Exception as e:
        print(f"Failed to query indicators for {owner}: {e}")

# Normalize and clean results
if final_results:
    # Extract and normalize data only if 'data' key exists and contains 'summary'
    normalized_data = []
    for result in final_results:
        if 'data' in result:
            for item in result['data']:
                if 'summary' in item:
                    normalized_data.append(item)

    if normalized_data:
        observed_src = pd.json_normalize(normalized_data)
        observed_src['indicator'] = observed_src['summary'].str.split(' ', expand=True)[0].str.strip()
        observed_src = observed_src.drop_duplicates(subset='indicator', keep='first')  # Remove duplicates based on 'indicator'
        observed_src = observed_src[observed_src['lastObserved'] >= start]
        print(f"\nRetrieved {len(observed_src)} unique observed indicators.")
    else:
        print("\nNo valid indicators with 'summary' key retrieved.")
else:
    print("\nNo indicators retrieved.")

display(observed_src)


Querying owner: HTOC Org

Retrieved 764 unique observed indicators.


,id,dateAdded,ownerId,ownerName,webLink,type,lastModified,rating,confidence,description,...,whoisActive,md5,sha1,sha256,size,text,address,Block,url,indicator
0,5629499565000772,2025-08-22T14:49:15Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-10-09T17:54:35Z,3.0,74.0,TASK0912447,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,185.241.208.71
3,4272777,2022-11-30T16:10:22Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-10-09T17:53:28Z,5.0,0.0,Emotet IOC,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,88.212.202.52
7,6755399459078660,2025-06-25T17:45:55Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-10-09T17:53:22Z,3.0,65.0,TASK0891295,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,64.62.197.101
8,5629499554313262,2025-06-11T14:46:29Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-10-09T17:53:22Z,3.0,64.0,RITM0588707 / TASK0886419,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,193.163.125.68
9,5629499542036631,2025-05-28T19:30:47Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-10-09T17:53:22Z,3.0,61.0,RITM0584685 / TASK0881439,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,193.163.125.127
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9975,6755399465229313,2025-07-28T17:15:43Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-10-06T15:34:56Z,3.0,70.0,TASK0902923,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,205.210.31.53
9976,6755399465229490,2025-07-28T17:33:48Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-10-06T15:34:55Z,3.0,70.0,TASK0902923,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,65.49.1.115
9977,6755399465229295,2025-07-28T17:15:25Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-10-06T15:34:55Z,3.0,70.0,TASK0902923,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,198.235.24.202
9980,6755399465229321,2025-07-28T17:15:49Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-10-06T15:34:50Z,3.0,70.0,TASK0902923,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,205.210.31.91


In [50]:
import os
import pandas as pd
from datetime import datetime, timedelta

# Base file path with placeholder for date
base_path = r"Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d{date}.csv"
#base_path = r"C:\Users\jaskew\Documents\project_repository\data\raw\ObservationDataFiles\htoc_opdiv_obs_d{date}.csv"
date_format = "%Y%m%d"

def get_file_paths(base_path, days=3):
    """ Generate file paths for the last `days` days using list comprehension. """
    today = datetime.utcnow()
    dates_to_pull = [(today - timedelta(days=i)).strftime(date_format) for i in range(days)]
    
    # Construct file paths
    file_paths = [base_path.format(date=dt) for dt in dates_to_pull]
    
    # Filter for existing files
    existing_files = [file_path for file_path in file_paths if os.path.exists(file_path)]
    
    if not existing_files:
        print("No files found for the specified date range.")
    else:
        print(f"Files to be loaded: {existing_files}")
    
    return existing_files

def load_observed_data(file_paths):
    """ Load and concatenate observed data from multiple files. """
    data_frames = []

    for file_path in file_paths:
        try:
            df = pd.read_csv(file_path)
            data_frames.append(df)
        except Exception as e:
            print(f"Error reading file {file_path}: {e}")
    
    # Concatenate data
    if data_frames:
        observed_data_df = pd.concat(data_frames, ignore_index=True)
        print(f"Loaded data from {len(data_frames)} files.")
    else:
        observed_data_df = pd.DataFrame()

    return observed_data_df

# Example Usage:
# Fetch file paths for the last 3 days
file_paths = get_file_paths(base_path, days=3)

# Load observed data
observed_data_df = load_observed_data(file_paths)



Files to be loaded: ['Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20251009.csv', 'Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20251008.csv', 'Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20251007.csv']


C:\Users\jaskew\AppData\Local\Temp\ipykernel_5416\564055639.py:12: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  today = datetime.utcnow()


Loaded data from 3 files.


In [51]:
import pandas as pd
from datetime import timedelta
import warnings

warnings.simplefilter(action='ignore', category=pd.errors.SettingWithCopyWarning)

# ═══════════════════════════════════════════════════════════════════════════════
# CONFIGURATION & SETUP
# ═══════════════════════════════════════════════════════════════════════════════

# Time cutoffs
cutoff = pd.Timestamp.utcnow()
cutoff_naive = cutoff.tz_convert(None)

# Required columns validation
required_columns = ['indicator', 'OpDiv', 'obs_date']
missing_columns = [c for c in required_columns if c not in observed_data_df.columns]
if missing_columns:
    raise ValueError(f"Missing columns in ProcessedObservedData.csv: {missing_columns}")

# ═══════════════════════════════════════════════════════════════════════════════
# HELPER FUNCTIONS
# ═══════════════════════════════════════════════════════════════════════════════

def process_tag_row(row, observed_src):
    """Process a single row from observed_src to extract and filter tags."""
    tags_data = row.get('tags.data')
    if not isinstance(tags_data, list):
        return None

    # Normalize the tags for the current row
    tags_df = pd.json_normalize(tags_data)

    # Ensure string and apply VA rename before filtering
    tags_df['name'] = (
        tags_df['name']
        .astype(str)
        .str.replace('VA CSOC CTS Splunk', 'VA Splunk API', regex=False)
    )

    # Skip if all associated groups are Adversary
    if 'associatedGroups.data' in observed_src.columns:
        ag_data = row.get('associatedGroups.data')
        if isinstance(ag_data, list) and len(ag_data) > 0:
            groups_df = pd.json_normalize(ag_data)
            if 'type' in groups_df.columns and set(groups_df['type']) == {'Adversary'}:
                return None

    # All tag names (for all_tags)
    all_tags_list = tags_df['name'].tolist()

    # Filter for API tags
    api_tags = tags_df[tags_df['name'].str.contains('API', case=False, na=False)].copy()
    if api_tags.empty:
        return None

    # Add metadata columns
    meta_cols = [
        'summary', 'observations', 'description', 'type',
        'dateAdded', 'lastModified', 'lastObserved', 'webLink',
        'rating', 'confidence', 'id', 'associatedGroups.data'
    ]
    for col in meta_cols:
        api_tags[col] = [row.get(col)] * len(api_tags)

    # Attach all tags list
    if len(api_tags) > 0:
        api_tags['all_tags'] = [all_tags_list] * len(api_tags)

    return api_tags

def map_observed_dates(filtered_tags, observed_data_df):
    """Map observed dates from observed_data_df to filtered_tags."""
    if filtered_tags.empty:
        return filtered_tags
    
    # Clean name -> match OpDiv (remove trailing ' Splunk API')
    filtered_tags['cleaned_name'] = filtered_tags['name'].str.replace(r'\s+Splunk API$', '', regex=True)
    filtered_tags['observed_date'] = pd.NaT
    
    # Ensure observed_data_df obs_date is datetime (naive)
    observed_data_df['obs_date'] = pd.to_datetime(observed_data_df['obs_date'], errors='coerce')

    for idx, r in filtered_tags.iterrows():
        summary = r.get('summary')
        cleaned_name = r.get('cleaned_name')
        if pd.isna(summary) or pd.isna(cleaned_name):
            continue
        match = observed_data_df[
            (observed_data_df['indicator'] == summary) &
            (observed_data_df['OpDiv'] == cleaned_name)
        ]
        if not match.empty:
            filtered_tags.at[idx, 'observed_date'] = match['obs_date'].iloc[0]

    filtered_tags['observed_date'] = pd.to_datetime(filtered_tags['observed_date'], errors='coerce')
    
    # Drop helper column
    filtered_tags.drop(columns=['cleaned_name'], inplace=True, errors='ignore')
    
    return filtered_tags

def apply_filters_and_grouping(filtered_tags, cutoff_naive):
    """Apply time filters, partner grouping, and other filtering criteria."""
    if filtered_tags.empty:
        return pd.DataFrame()
    
    # Last 2 days by observed_date (naive)
    recent_tags = filtered_tags[filtered_tags['observed_date'] >= cutoff_naive - timedelta(days=2)].copy()

    if recent_tags.empty:
        return recent_tags
    
    # Partner extraction and grouping
    recent_tags['partner'] = recent_tags['name'].str.replace(' Splunk API', '', regex=False)

    partner_groups = (
        recent_tags.groupby('summary')['partner']
        .agg(['nunique', lambda s: ', '.join(sorted(set(s.dropna())))]).reset_index()
        .rename(columns={'nunique': 'partner_count', '<lambda_0>': 'partners'})
    )

    recent_tags = recent_tags.merge(partner_groups, on='summary', how='left')

    # Additional filters
    recent_tags = recent_tags[recent_tags['partner_count'] >= 2]

    # Deduplicate by summary
    recent_tags = recent_tags.drop_duplicates(subset='summary', keep='first')

    # Drop unneeded columns if present
    cols_to_drop = [
        'techniqueId', 'tactics.data', 'tactics.count',
        'platforms.data', 'platforms.count', 'partner', 'name'
    ]
    recent_tags = recent_tags.drop(columns=[c for c in cols_to_drop if c in recent_tags.columns], errors='ignore')

    # Remove rows where all_tags contains unwanted markers
    if 'all_tags' in recent_tags.columns:
        recent_tags = recent_tags[~recent_tags['all_tags'].apply(lambda x: isinstance(x, list) and 'I&W' in x)]
        recent_tags = recent_tags[~recent_tags['all_tags'].apply(lambda x: isinstance(x, list) and 'htoc_wl' in x)]

    return recent_tags

def extract_group_ids(recent_tags):
    """Extract group IDs from associatedGroups.data."""
    if 'associatedGroups.data' in recent_tags.columns:
        recent_tags['group_id'] = recent_tags['associatedGroups.data'].apply(
            lambda x: x[0]['id'] if isinstance(x, list) and len(x) > 0 and 'id' in x[0] else None
        )
        # Convert group_id to string type and remove trailing decimals if it exists
        if 'group_id' in recent_tags.columns:
            recent_tags['group_id'] = recent_tags['group_id'].apply(
                lambda x: str(int(float(x))) if pd.notna(x) and str(x) != 'None' else x
            ).astype(str)
    return recent_tags

# ═══════════════════════════════════════════════════════════════════════════════
# MAIN PROCESSING PIPELINE
# ═══════════════════════════════════════════════════════════════════════════════

print("Starting tag processing pipeline...")

# Step 1: Process tags from observed_src
print("Processing tags from observed_src...")
all_filtered = []

for _, row in observed_src.iterrows():
    processed_tags = process_tag_row(row, observed_src)
    if processed_tags is not None:
        all_filtered.append(processed_tags)

# Step 2: Create filtered_tags DataFrame
print("Creating filtered_tags DataFrame...")
filtered_tags = pd.concat(all_filtered, ignore_index=True) if all_filtered else pd.DataFrame()

if not filtered_tags.empty:
    # Ensure datetime columns
    filtered_tags['lastObserved'] = pd.to_datetime(filtered_tags['lastObserved'], errors='coerce', utc=True)
    filtered_tags['dateAdded'] = pd.to_datetime(filtered_tags['dateAdded'], errors='coerce', utc=True)
    print(f"Created filtered_tags with {len(filtered_tags)} rows")
else:
    print("No filtered tags found")

# Step 3: Map observed dates
print("Mapping observed dates...")
filtered_tags = map_observed_dates(filtered_tags, observed_data_df)

# Step 4: Apply filters and grouping
print("Applying filters and partner grouping...")
recent_tags = apply_filters_and_grouping(filtered_tags, cutoff_naive)

# Step 5: Extract group IDs
print("Extracting group IDs...")
recent_tags = extract_group_ids(recent_tags)

# Final summary
print(f"Processing complete! Final dataset shape: {recent_tags.shape}")
if not recent_tags.empty:
    print(f"Partners represented: {recent_tags['partners'].nunique()} unique partner combinations")

display(recent_tags)

Starting tag processing pipeline...
Processing tags from observed_src...
Creating filtered_tags DataFrame...
Created filtered_tags with 5844 rows
Mapping observed dates...
Applying filters and partner grouping...
Extracting group IDs...
Processing complete! Final dataset shape: (518, 18)
Partners represented: 82 unique partner combinations


,id,description,lastUsed,summary,observations,type,dateAdded,lastModified,lastObserved,webLink,rating,confidence,associatedGroups.data,all_tags,observed_date,partner_count,partners,group_id
2,6755399459078660,TASK0891295,2025-10-09T16:20:09Z,64.62.197.101,962869,Address,2025-06-25 17:45:55+00:00,2025-10-09T17:53:22Z,2025-10-09 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,3.0,65.0,NaN,"[DHA Splunk API, OS Splunk API, FDA Splunk API...",2025-10-09,8,"CMS, DHA, FDA, HHS, HRSA, IHS, OS, VA",nan
10,5629499554313262,RITM0588707 / TASK0886419,2025-10-09T16:20:09Z,193.163.125.68,2405163,Address,2025-06-11 14:46:29+00:00,2025-10-09T17:53:22Z,2025-10-09 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,3.0,64.0,NaN,"[DHA Splunk API, OS Splunk API, FDA Splunk API...",2025-10-09,9,"CMS, DHA, FDA, HHS, HRSA, IHS, NIH, OS, VA",nan
19,5629499542036631,RITM0584685 / TASK0881439,2025-10-09T14:51:46Z,193.163.125.127,2627245,Address,2025-05-28 19:30:47+00:00,2025-10-09T17:53:22Z,2025-10-09 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,3.0,61.0,NaN,"[DHA Splunk API, OS Splunk API, FDA Splunk API...",2025-10-09,7,"CMS, FDA, HHS, HRSA, NIH, OS, VA",nan
26,5629499571030621,INC9264879,2025-10-09T14:51:46Z,36.213.200.104,11062,Address,2025-10-04 16:08:58+00:00,2025-10-09T17:36:40Z,2025-10-09 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,3.0,80.0,NaN,"[OS Splunk API, FDA Splunk API, suspicious, CM...",2025-10-09,3,"CMS, FDA, OS",nan
29,5629499571030620,INC9264879,2025-10-09T14:51:46Z,116.124.133.221,47405,Address,2025-10-04 16:08:57+00:00,2025-10-09T17:36:40Z,2025-10-09 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,3.0,80.0,NaN,"[OS Splunk API, FDA Splunk API, suspicious, CM...",2025-10-09,4,"CMS, FDA, OS, VA",nan
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3673,6755399465229541,TASK0902923,2025-10-09T16:20:09Z,205.210.31.141,1808100,Address,2025-07-28 17:34:21+00:00,2025-10-06T15:34:56Z,2025-10-09 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,3.0,70.0,NaN,"[DHA Splunk API, OS Splunk API, FDA Splunk API...",2025-10-09,8,"CMS, DHA, FDA, HRSA, IHS, NIH, OS, VA",nan
3681,6755399465229313,TASK0902923,2025-10-09T16:20:09Z,205.210.31.53,9504191,Address,2025-07-28 17:15:43+00:00,2025-10-06T15:34:56Z,2025-10-09 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,3.0,70.0,NaN,"[DHA Splunk API, OS Splunk API, FDA Splunk API...",2025-10-09,7,"CMS, DHA, FDA, HRSA, IHS, OS, VA",nan
3688,6755399465229490,TASK0902923,2025-10-09T16:20:09Z,65.49.1.115,1050471,Address,2025-07-28 17:33:48+00:00,2025-10-06T15:34:55Z,2025-10-09 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,3.0,70.0,NaN,"[DHA Splunk API, OS Splunk API, FDA Splunk API...",2025-10-09,7,"CMS, DHA, FDA, HRSA, IHS, OS, VA",nan
3695,6755399465229295,TASK0902923,2025-10-09T16:20:09Z,198.235.24.202,10331171,Address,2025-07-28 17:15:25+00:00,2025-10-06T15:34:55Z,2025-10-09 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,3.0,70.0,NaN,"[DHA Splunk API, OS Splunk API, FDA Splunk API...",2025-10-09,8,"CMS, DHA, FDA, HRSA, IHS, NIH, OS, VA",nan


In [64]:
import pandas as pd

# Extract unique indicators from recent_tags
indicators = recent_tags['summary'].unique()

# Initialize a list to store attribute data
attributes_data = []

# Track indicators with no attributes
indicators_with_no_attributes = []

# Iterate over unique indicators
for indicator in indicators:
    try:
        ro.set_http_method('GET')
        ro.set_request_uri(f'/v3/indicators/{indicator}?fields=attributes&resultStart=0&resultLimit=1000')
        response = tc.api_request(ro)

        if response.headers.get('content-type') == 'application/json':
            data = response.json().get('data', {})
            attributes = data.get('attributes', {}).get('data', [])

            if not attributes:
                print(f"No attributes found for indicator: {indicator}")
                # Track indicators with no attributes
                indicators_with_no_attributes.append(indicator)
            else:
                for attr in attributes:
                    attr.update({
                        'id': data.get('id'),
                        'summary': data.get('summary'),
                        'type': data.get('type'),
                        'ownerName': data.get('ownerName')
                    })
                    attributes_data.append(attr)
        # Suppress non-JSON and known 400 error responses
    except Exception as e:
        # Suppress error output, including known 400 error
        if hasattr(e, 'response') and getattr(e.response, 'status_code', None) == 400:
            continue
        if "Status Code: 400" in str(e):
            continue
        pass

# Create attributes 
attributes_observed_src = pd.json_normalize(attributes_data)

# Un-nest 'createdBy' and filter out 'SOAR' entries
if not attributes_observed_src.empty and attributes_observed_src['createdBy.lastName'].notnull().any():
    attributes_observed_src = attributes_observed_src[attributes_observed_src['createdBy.lastName'] != 'SOAR']

# Drop duplicates based on 'id' to keep unique attribute records
attributes_observed_src = attributes_observed_src.drop_duplicates(subset='id').reset_index(drop=True)

# Filter recent_tags for indicators that had attributes
filtered_with_attrs = recent_tags[recent_tags['summary'].isin(attributes_observed_src['summary'])]

# Filter recent_tags for indicators that had no attributes
no_attrs_df = recent_tags[recent_tags['summary'].isin(indicators_with_no_attributes)]

# Combine both and remove duplicates based on 'summary'
filtered_recent_tags = pd.concat([filtered_with_attrs, no_attrs_df], ignore_index=True)
filtered_recent_tags = filtered_recent_tags.drop_duplicates(subset='summary').reset_index(drop=True)
display(filtered_recent_tags)


No attributes found for indicator: 37.19.207.37
No attributes found for indicator: 172.98.33.195


,id,description,lastUsed,summary,observations,type,dateAdded,lastModified,lastObserved,webLink,rating,confidence,associatedGroups.data,all_tags,observed_date,partner_count,partners,group_id
0,6755399459033712,The IP is hosted by Linode (Akamai Connected C...,2025-10-09T16:20:09Z,172.233.190.104,98719141,Address,2025-06-16 17:35:08+00:00,2025-10-09T17:34:22Z,2025-10-09 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,2.0,78.0,NaN,"[DHA Splunk API, Cloud, OS Splunk API, FDA Spl...",2025-10-09,5,"CMS, DHA, HHS, HRSA, NIH",nan
1,6755399459033709,"IP 45.33.69.253 is a high-confidence, high-ris...",2025-10-09T16:23:15Z,45.33.69.253,102464235,Address,2025-06-16 17:35:07+00:00,2025-10-09T17:34:22Z,2025-10-09 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,1.0,38.0,NaN,"[DHA Splunk API, Cloud, OS Splunk API, FDA Spl...",2025-10-09,4,"CMS, HHS, HRSA, NIH",nan
2,6755399459033706,IP address 172.233.195.143 has been flagged fo...,2025-10-09T16:23:15Z,172.233.195.143,90817880,Address,2025-06-16 17:35:05+00:00,2025-10-09T17:34:22Z,2025-10-09 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,1.0,48.0,NaN,"[DHA Splunk API, OS Splunk API, FDA Splunk API...",2025-10-09,4,"CMS, HHS, HRSA, NIH",nan
3,6755399459033703,IP address 172.232.188.177 has been identified...,2025-10-09T16:23:15Z,172.232.188.177,101520385,Address,2025-06-16 17:35:02+00:00,2025-10-09T17:34:22Z,2025-10-09 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,1.0,41.0,NaN,"[DHA Splunk API, Cloud, OS Splunk API, FDA Spl...",2025-10-09,4,"CMS, HHS, HRSA, NIH",nan
4,6755399459033700,172.234.207.202 is a critically rated maliciou...,2025-10-09T16:23:15Z,172.234.207.202,104833938,Address,2025-06-16 17:35:01+00:00,2025-10-09T17:34:22Z,2025-10-09 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,1.0,68.0,NaN,"[DHA Splunk API, Cloud, OS Splunk API, FDA Spl...",2025-10-09,4,"CMS, HHS, HRSA, NIH",nan
5,5629499555060654,IP address 172.234.246.109 has been flagged fo...,2025-10-09T16:23:15Z,172.234.246.109,91902099,Address,2025-06-16 17:35:07+00:00,2025-10-09T17:34:22Z,2025-10-09 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,1.0,2.0,NaN,"[DHA Splunk API, Cloud, OS Splunk API, FDA Spl...",2025-10-09,4,"CMS, HHS, HRSA, NIH",nan
6,5629499555060653,66.175.208.166 is a high-risk IP currently hos...,2025-10-09T16:20:09Z,66.175.208.166,105435530,Address,2025-06-16 17:35:06+00:00,2025-10-09T17:34:22Z,2025-10-09 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,1.0,73.0,NaN,"[DHA Splunk API, Cloud, OS Splunk API, FDA Spl...",2025-10-09,5,"CMS, DHA, HHS, HRSA, NIH",nan
7,6755399457468485,A CMS user received a phishing email that cont...,2025-10-09T16:23:15Z,link.mail.beehiiv.com,109244,Host,2025-06-06 13:12:09+00:00,2025-10-09T17:15:04Z,2025-10-09 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,3.0,45.0,"[{'id': 6755399449000058, 'dateAdded': '2025-0...","[DHA Splunk API, OS Splunk API, FDA Splunk API...",2025-10-09,3,"CMS, HRSA, NIH",6755399449000058
8,5629499554313271,IP address 104.237.151.205 has been heavily ob...,2025-10-09T16:20:09Z,104.237.151.205,107059729,Address,2025-06-11 14:47:15+00:00,2025-10-09T17:13:19Z,2025-10-09 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,1.0,68.0,NaN,"[DHA Splunk API, Cloud, OS Splunk API, FDA Spl...",2025-10-09,5,"CMS, DHA, HHS, HRSA, NIH",nan
9,4809962,Sharing malicious indicators flagged by Virust...,2025-10-09T16:23:15Z,165.22.54.16,1947152,Address,2024-08-08 18:17:34+00:00,2025-10-09T16:36:24Z,2025-10-09 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,3.0,24.0,"[{'id': 447870, 'dateAdded': '2024-08-08T18:17...","[Indicators, Malicious Activity, OS Splunk API...",2025-10-09,2,"CMS, HHS",447870


In [65]:
import os, io, csv
import pandas as pd

# ── Path to the reported indicators file ──────────────────────────────────────
reported_indicators_path = r"Z:\HTOC\HTOC Reports\I&W Reports\5. I&W Staging\Reported Indicators\indicators.csv"

# ── Robust loader: returns (list_of_nonempty_values, column_used) ─────────────
def _load_indicator_values(path: str, col: str = "Indicator"):
    if not os.path.exists(path):
        raise FileNotFoundError(f"File not found: {path}")

    # 1) Try pandas with delimiter sniffing (no noisy warnings)
    try:
        df = pd.read_csv(
            path,
            sep=None,
            engine="python",
            encoding="utf-8-sig",
            dtype=str
        )
        if col not in df.columns:
            # Case-insensitive fallback
            ci_match = [c for c in df.columns if c.strip().lower() == col.lower()]
            if ci_match:
                col = ci_match[0]
            else:
                raise KeyError(f"Column '{col}' not found. Columns: {list(df.columns)}")

        ser = (
            df[col]
            .astype(str)
            .str.strip()
            .replace("", pd.NA)
            .dropna()
        )
        return ser.tolist(), col

    except Exception:
        # 2) Fallback to csv.Sniffer for very messy files
        with open(path, "rb") as fh:
            raw = fh.read()

        text = None
        for enc in ("utf-8-sig", "utf-8", "latin-1"):
            try:
                text = raw.decode(enc)
                break
            except UnicodeDecodeError:
                continue
        if text is None:
            raise UnicodeDecodeError("utf-8/latin-1", b"", 0, 1, "Could not decode file")

        sample = text[:4096]
        try:
            dialect = csv.Sniffer().sniff(sample)
            has_header = csv.Sniffer().has_header(sample)
        except Exception:
            dialect = csv.excel
            dialect.delimiter = ","
            dialect.skipinitialspace = True
            has_header = True

        if has_header:
            reader = csv.DictReader(io.StringIO(text), dialect=dialect)
            header = [h.strip().lstrip("\ufeff") for h in reader.fieldnames or []]
            # normalize to located column name actually present
            if col not in header:
                ci_match = [h for h in header if h.lower() == col.lower()]
                if ci_match:
                    col = ci_match[0]
                else:
                    raise KeyError(f"Column '{col}' not found. Columns: {header}")

            vals = []
            for row in reader:
                if not row:
                    continue
                v = (row.get(col) or "").strip()
                if v:
                    vals.append(v)
            return vals, col
        else:
            # no header, assume first column is Indicator
            rdr = csv.reader(io.StringIO(text), dialect=dialect)
            vals = []
            for parts in rdr:
                if parts:
                    v = (parts[0] or "").strip()
                    if v:
                        vals.append(v)
            return vals, col  # col label remains as requested

# ── Load reported indicators and filter out already-reported indicators ───────
try:
    values, col_name = _load_indicator_values(reported_indicators_path, col="Indicator")
    print(f"Loaded reported indicators — non-empty rows in '{col_name}': {len(values)}")

    # Unique set for filtering
    reported_set = set(values)

    # Filter out already-reported indicators from filtered_recent_tags
    if 'filtered_recent_tags' in globals() and not filtered_recent_tags.empty and reported_set:
        before_count = len(filtered_recent_tags)
        filtered_recent_tags = filtered_recent_tags[
            ~filtered_recent_tags['summary'].astype(str).isin(reported_set)
        ].reset_index(drop=True)
        after_count = len(filtered_recent_tags)
        print(f"Removed {before_count - after_count} already-reported indicators")
        print(f"Final filtered dataset: {after_count} indicators")
    elif 'filtered_recent_tags' not in globals():
        print("Warning: filtered_recent_tags not found - run previous cells first")
    else:
        print("No filtering applied (empty dataset or no reported indicators).")

except Exception as e:
    print(f"[ERROR] {e}")

# ── Display final filtered results ────────────────────────────────────────────
if 'filtered_recent_tags' in globals():
    display(filtered_recent_tags)
else:
    print("No filtered_recent_tags to display")


Loaded reported indicators — non-empty rows in 'Indicator': 249
Removed 1 already-reported indicators
Final filtered dataset: 31 indicators


,id,description,lastUsed,summary,observations,type,dateAdded,lastModified,lastObserved,webLink,rating,confidence,associatedGroups.data,all_tags,observed_date,partner_count,partners,group_id
0,6755399459033712,The IP is hosted by Linode (Akamai Connected C...,2025-10-09T16:20:09Z,172.233.190.104,98719141,Address,2025-06-16 17:35:08+00:00,2025-10-09T17:34:22Z,2025-10-09 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,2.0,78.0,NaN,"[DHA Splunk API, Cloud, OS Splunk API, FDA Spl...",2025-10-09,5,"CMS, DHA, HHS, HRSA, NIH",nan
1,6755399459033709,"IP 45.33.69.253 is a high-confidence, high-ris...",2025-10-09T16:23:15Z,45.33.69.253,102464235,Address,2025-06-16 17:35:07+00:00,2025-10-09T17:34:22Z,2025-10-09 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,1.0,38.0,NaN,"[DHA Splunk API, Cloud, OS Splunk API, FDA Spl...",2025-10-09,4,"CMS, HHS, HRSA, NIH",nan
2,6755399459033706,IP address 172.233.195.143 has been flagged fo...,2025-10-09T16:23:15Z,172.233.195.143,90817880,Address,2025-06-16 17:35:05+00:00,2025-10-09T17:34:22Z,2025-10-09 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,1.0,48.0,NaN,"[DHA Splunk API, OS Splunk API, FDA Splunk API...",2025-10-09,4,"CMS, HHS, HRSA, NIH",nan
3,6755399459033703,IP address 172.232.188.177 has been identified...,2025-10-09T16:23:15Z,172.232.188.177,101520385,Address,2025-06-16 17:35:02+00:00,2025-10-09T17:34:22Z,2025-10-09 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,1.0,41.0,NaN,"[DHA Splunk API, Cloud, OS Splunk API, FDA Spl...",2025-10-09,4,"CMS, HHS, HRSA, NIH",nan
4,6755399459033700,172.234.207.202 is a critically rated maliciou...,2025-10-09T16:23:15Z,172.234.207.202,104833938,Address,2025-06-16 17:35:01+00:00,2025-10-09T17:34:22Z,2025-10-09 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,1.0,68.0,NaN,"[DHA Splunk API, Cloud, OS Splunk API, FDA Spl...",2025-10-09,4,"CMS, HHS, HRSA, NIH",nan
5,5629499555060654,IP address 172.234.246.109 has been flagged fo...,2025-10-09T16:23:15Z,172.234.246.109,91902099,Address,2025-06-16 17:35:07+00:00,2025-10-09T17:34:22Z,2025-10-09 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,1.0,2.0,NaN,"[DHA Splunk API, Cloud, OS Splunk API, FDA Spl...",2025-10-09,4,"CMS, HHS, HRSA, NIH",nan
6,5629499555060653,66.175.208.166 is a high-risk IP currently hos...,2025-10-09T16:20:09Z,66.175.208.166,105435530,Address,2025-06-16 17:35:06+00:00,2025-10-09T17:34:22Z,2025-10-09 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,1.0,73.0,NaN,"[DHA Splunk API, Cloud, OS Splunk API, FDA Spl...",2025-10-09,5,"CMS, DHA, HHS, HRSA, NIH",nan
7,5629499554313271,IP address 104.237.151.205 has been heavily ob...,2025-10-09T16:20:09Z,104.237.151.205,107059729,Address,2025-06-11 14:47:15+00:00,2025-10-09T17:13:19Z,2025-10-09 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,1.0,68.0,NaN,"[DHA Splunk API, Cloud, OS Splunk API, FDA Spl...",2025-10-09,5,"CMS, DHA, HHS, HRSA, NIH",nan
8,4809962,Sharing malicious indicators flagged by Virust...,2025-10-09T16:23:15Z,165.22.54.16,1947152,Address,2024-08-08 18:17:34+00:00,2025-10-09T16:36:24Z,2025-10-09 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,3.0,24.0,"[{'id': 447870, 'dateAdded': '2024-08-08T18:17...","[Indicators, Malicious Activity, OS Splunk API...",2025-10-09,2,"CMS, HHS",447870
9,6755399460008411,"Details\nIn late June 2025, Iran-aligned hackt...",2025-10-09T16:20:09Z,200.31.12.168,159,Address,2025-07-02 12:05:37+00:00,2025-10-09T14:14:16Z,2025-10-08 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,2.0,60.0,"[{'id': 5629499544001057, 'dateAdded': '2025-0...","[INDICATOR NOTICE 25178.1, Mr Hamza Group, T-S...",2025-10-08,2,"DHA, VA",5629499544001057


In [66]:
from datetime import datetime, UTC

output_path = 'Z:\\HTOC\\HTOC Reports\\I&W Reports\\5. I&W Staging\\Spreadsheet\\Expanded'

# Generate filename with today's date in YYYYMMDD format
today_str = datetime.now(UTC).strftime("%Y%m%d")
excel_filename = f"expanded_indicators_{today_str}.xlsx"
excel_path = os.path.join(output_path, excel_filename)

# Convert all timezone-aware datetime columns to naive (remove timezone info)
for col in filtered_recent_tags.select_dtypes(include=['datetimetz']).columns:
    filtered_recent_tags[col] = filtered_recent_tags[col].dt.tz_localize(None)

# Drop the 'description' column if it exists
if 'description' in filtered_recent_tags.columns:
    filtered_recent_tags = filtered_recent_tags.drop(columns=['description'])

filtered_recent_tags.to_excel(excel_path, index=False)
print(f"Saved to {excel_path}")

Saved to Z:\HTOC\HTOC Reports\I&W Reports\5. I&W Staging\Spreadsheet\Expanded\expanded_indicators_20251009.xlsx
